# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [341]:
%pip install pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [342]:
# importing necessary libraries

import pandas as pd
import matplotlib.pyplot as plt

In [343]:
df = pd.read_csv("/Users/Anika/ironhack/week2/lab-functions-en/tic-tac-toe.csv") # reads csv file using pandas read_csv function

In [344]:
# importing required libraries required for deep learning 

import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout 
from tensorflow.keras.optimizers import Adam 



In [345]:
# lets check the system and the version

import sys
import platform 

print("The python version is:", sys.version)
print("Python executable:", sys.executable)
print("Architecture:", platform.machine())
print("Operating system:", platform.platform())

The python version is: 3.12.16 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 20:15:26) [Clang 20.1.8 ]
Python executable: /opt/homebrew/anaconda3/envs/tictactoe/bin/python
Architecture: arm64
Operating system: macOS-15.5-arm64-arm-64bit


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [346]:
# lets inspect the dataset before building the deep learning model

print(df.head()) # first 5 rows

print(df.shape) # dimensions of the data

  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
(958, 10)


In [347]:
print(df.info()) 

print(df.dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   TL      958 non-null    str  
 1   TM      958 non-null    str  
 2   TR      958 non-null    str  
 3   ML      958 non-null    str  
 4   MM      958 non-null    str  
 5   MR      958 non-null    str  
 6   BL      958 non-null    str  
 7   BM      958 non-null    str  
 8   BR      958 non-null    str  
 9   class   958 non-null    bool 
dtypes: bool(1), str(9)
memory usage: 68.4 KB
None
TL        str
TM        str
TR        str
ML        str
MM        str
MR        str
BL        str
BM        str
BR        str
class    bool
dtype: object


In [348]:
print(df.isnull().sum()) # are there missing values?

print(df["class"].value_counts()) # are there unique classes?

TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64
class
True     626
False    332
Name: count, dtype: int64


In [349]:
# checking class imbalance

percentage_true = (df["class"].sum()/ len(df))*100 
print(percentage_true)

65.34446764091858


In [350]:
# 65% of the classes are True and 35% are false, there is a moderate class imbalance


In [351]:
# Data preprocessing : convert categorical variables to numeric

# 1. lets use one-hot encoding to convert categorical values into numerical ones so that the deep learning can perform appropriate mathematical operations

# we create another dataframe called df_encoded to preserve the original dataframe

df_encoded = pd.get_dummies(df, columns = ["TL", "TM", "TR", "ML", "MM", "MR","BL", "BM", "BR"], drop_first = False, dtype = int) # drop_first = False has been assigned for the deep learning approach as we want to retain all information, even if is redundant 

#2. initialize x and y variables 

X = df_encoded.drop(columns = ["class"]) 
y = df_encoded["class"] 






In [352]:
# lets install scikit-learn

%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [353]:
from sklearn.model_selection import train_test_split 

# 80% training and 20% testing 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42) 

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(766, 27)
(766,)
(192, 27)
(192,)


In [354]:
# we have 958 observations and 27 input features corresponding to the 9 columns(after one-hot encoding) and 2 output classes 

# lets initialize the hyperparameters- chosen before training 
EPOCHS = 90
BATCH_SIZE = 72
NUM_CLASSES = 2

In [355]:
# lets build a deep learning model
# 1. first, a sequential model, which means that layers are arranged one after another

model = Sequential() # this is an empty neural network

#2. lets add the output and the hidden layers

model.add(Dense(32, activation = "relu", input_shape=(27,))) # first hidden layer 
model.add(Dense(64, activation = "relu", input_shape = (27,))) # 2nd hidden layer 


#3. lets add the output layer

model.add(Dense(NUM_CLASSES, activation = "softmax"))
print(model.summary()) # check every layer with the neuron number and the parameters(weights and biases)



/opt/homebrew/anaconda3/envs/tictactoe/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_56 (Dense)                │ (None, 32)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_57 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_58 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,138 (12.26 KB)

 Trainable params: 3,138 (12.26 KB)

 Non-trainable params: 0 (0.00 B)

None


In [356]:
# converting boolean values to integers

y_train = y_train.astype("int32")
y_test = y_test.astype("int32") 

# compiling the model
model.compile(optimizer = Adam(learning_rate = 0.001), loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])

In [357]:
# fit the model

history = model.fit(X_train, y_train, batch_size = BATCH_SIZE,epochs=EPOCHS, validation_split = 0.2 )

Epoch 1/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6503 - loss: 0.6484 - val_accuracy: 0.6753 - val_loss: 0.6140
Epoch 2/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6520 - loss: 0.6227 - val_accuracy: 0.6948 - val_loss: 0.5994
Epoch 3/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6618 - loss: 0.6003 - val_accuracy: 0.6948 - val_loss: 0.5832
Epoch 4/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6699 - loss: 0.5799 - val_accuracy: 0.7208 - val_loss: 0.5701
Epoch 5/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7075 - loss: 0.5580 - val_accuracy: 0.7532 - val_loss: 0.5582
Epoch 6/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7271 - loss: 0.5340 - val_accuracy: 0.7468 - val_loss: 0.5446
Epoch 7/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7533 - loss: 0.5109 - val_accuracy: 0.7403 - val_loss: 0.5350
Epoch 8/90
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7696 - loss: 0.4888 - val_accuracy: 0.7468 - val_loss: 0.5253
Epoch 9

In [362]:
# evaluate the model 

test_loss, test_accuracy = model.evaluate(X_test, y_test) 

print("Test loss:", test_loss) 
print("Test accuracy:", test_accuracy)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9688 - loss: 0.0600 
Test loss: 0.05997486785054207
Test accuracy: 0.96875


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [371]:
# save model

model.save("tictactoe_model.keras") 

from tensorflow.keras.models import load_model 

loaded_model = load_model("tictactoe_model.keras") # load model

In [372]:
# select 5 random test observations

random_samples = X_test.sample(5, random_state = 42)

In [373]:
%pip install numpy 

import numpy as np

^C
Note: you may need to restart the kernel to use updated packages.


In [375]:
# make predictions 

predictions = loaded_model.predict(random_samples) 

# argmax row by row converts probabilities of classes into 0 and 1

predicted_classes = predictions.argmax(axis = 1) 

actual_classes = y_test.loc[random_samples.index] 

# looping through actual classes and predicted classes 

for actual_class, predicted_class in zip(actual_classes, predicted_classes):
    print(f"Actual: {actual_class}, Predicted: {bool(predicted_class)}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
Actual: 0, Predicted: False
Actual: 1, Predicted: True
Actual: 1, Predicted: True
Actual: 1, Predicted: True
Actual: 0, Predicted: False


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [ ]:
# I have already achieved my goal of loss < 0.1 and accuracy greater than 0.95

**Which approach(es) did you find helpful to improve your model performance?**

In [361]:
# your answer here